In [5]:
import pandas as pd
import numpy as np
import os
import json
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from pathlib import Path
from tabgan.sampler import GANGenerator

os.makedirs("synthetic_extension", exist_ok=True)

#df_train = pd.read_csv("../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv")
#Use the whole dataset
df_train = pd.read_csv("../datasets/CICEVSE2024/synthetic_extension/EVSE-B-PowerCombined_filtered.csv")

In [6]:
txt = """
Real CSV path.
../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv

The four feature column names.
shunt_voltage, current_mA, power_mW, bus_voltage_V, State

The attack label column name. "Attack"

Which of the five columns are categorical. "Attack", "State"

Whether you want class-wise generation in separate loops per attack class, or one global generation pass followed by balancing.
one global generation

Whether you already have a train/test split file pair, because tabgan is designed around train_df and test_df
I already hav a test  dataset and need a train dataset with TabGAN
"""
print(txt)


Real CSV path.
../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv

The four feature column names.
shunt_voltage, current_mA, power_mW, bus_voltage_V, State

The attack label column name. "Attack"

Which of the five columns are categorical. "Attack", "State"

Whether you want class-wise generation in separate loops per attack class, or one global generation pass followed by balancing.
one global generation

Whether you already have a train/test split file pair, because tabgan is designed around train_df and test_df
I already hav a test  dataset and need a train dataset with TabGAN



In [1]:
from pathlib import Path
import pandas as pd
from tabgan.sampler import GANGenerator

# --------------------------------------------------
# Paths
# --------------------------------------------------
train_path = Path("../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv")
output_dir = Path("../datasets/CICEVSE2024/synthetic_extension")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Dataset configuration
# --------------------------------------------------
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]
target_col = "Attack"
cat_cols = ["State"]

target_total_rows = 34312
target_distribution = {
    "none": 0.2930,
    "Backdoor": 0.4312,
    "syn-flood": 0.2758
}

# --------------------------------------------------
# TabGAN variants
# --------------------------------------------------
'''
variant_configs = [
    {#original distribution, less aggressive sampling.
        "name": "baseline",
        "gen_x_times": 1.0, # wie viel Roh-samples insgesamt
        "pregeneration_frac": 2, # wie viel extra samples vor dem Filter, danach aussortieren.
        "is_post_process": True, # TURE ignor filter quantile -> closer to original distribution | False raw generated samples more untoched
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False, # extra filtering steps off -> more distribution 
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 300},
    },
    {
        "name": "variant_a",# A bit more diversity than baseline while still keeping filtering strict.
        "gen_x_times": 1.5,
        "pregeneration_frac": 2,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 200},
    },
    {
        "name": "variant_b",# Stronger coverage/diversity and more room for the filter to remove bad samples. 
        "gen_x_times": 2.0,
        "pregeneration_frac": 3,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 25, "epochs": 300},
    },
    {
        "name": "variant_c",#Maximum raw output retention, less constrained, potentially more diverse but also more noise/outliers. 
        "gen_x_times": 1.5,
        "pregeneration_frac": 3,
        "is_post_process": False,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 300},
    },
]
'''
variant_configs = [
    {
        "name": "best_variant_1",
        "gen_x_times": 0.75,
        "pregeneration_frac": 1,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.9999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 300, "patience": 25, "epochs": 100},
    },"""
    {
        "name": "best_variant_2",
        "gen_x_times": 1.5,
        "pregeneration_frac": 3,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 128, "patience": 20, "epochs": 200},
    },
    {
        "name": "best_variant_3",
        "gen_x_times": 1.0,
        "pregeneration_frac": 3,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 128, "patience": 20, "epochs": 200},
    },
    {
        "name": "best_variant_4",
        "gen_x_times": 1.0,
        "pregeneration_frac": 3,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 128, "patience": 20, "epochs": 100},
    },"""
]

# --------------------------------------------------
# Load and validate dataset
# --------------------------------------------------
df = pd.read_csv(train_path)
required_cols = feature_cols + [target_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy()
df["State"] = df["State"].astype(str)
df["Attack"] = df["Attack"].astype(str)

# --------------------------------------------------
# Compute target row count per class
# --------------------------------------------------
target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}
remainder = target_total_rows - sum(target_counts.values())

if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target counts:", target_counts)

# --------------------------------------------------
# Run class-wise TabGAN generation
# Each class is trained independently with its own model.
# --------------------------------------------------
summary_rows = []

for cfg in variant_configs:
    print(f"\n{'=' * 70}")
    print(f"Running TabGAN variant: {cfg['name']}")
    print(f"Configuration: {cfg}")
    print(f"{'=' * 70}")

    class_outputs = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining separate TabGAN for class: {attack_label}")

        # Select only rows from the current class
        df_class = df[df[target_col] == attack_label].copy()

        if df_class.empty:
            raise ValueError(f"No rows found for class '{attack_label}'.")

        if len(df_class) < 10:
            raise ValueError(f"Not enough rows for class '{attack_label}' to train a stable TabGAN.")

        # Encode the class target as a constant numeric value
        # A separate model is trained per class, so the target is constant here.
        df_class["Attack_enc"] = 0

        train_df = df_class[feature_cols].copy()
        target_df = df_class[["Attack_enc"]].copy()

        # Use the class-specific feature distribution as reference distribution
        test_df = train_df.copy()

        generator = GANGenerator(
            gen_x_times=cfg["gen_x_times"],
            cat_cols=cat_cols,
            pregeneration_frac=cfg["pregeneration_frac"],
            is_post_process=cfg["is_post_process"],
            bot_filter_quantile=cfg["bot_filter_quantile"],
            top_filter_quantile=cfg["top_filter_quantile"],
            gen_params=cfg["gen_params"]
        )

        synthetic_x, synthetic_y = generator.generate_data_pipe(
            train_df=train_df,
            target=target_df,
            test_df=test_df,
            deep_copy=True,
            only_adversarial=False,
            use_adversarial=cfg["use_adversarial"]
        )

        synthetic_df = pd.concat(
            [synthetic_x.reset_index(drop=True), synthetic_y.reset_index(drop=True)],
            axis=1
        )

        # Ensure column names are correct even if TabGAN returns unnamed target output
        if "Attack_enc" not in synthetic_df.columns:
            if synthetic_df.shape[1] == len(feature_cols) + 1:
                synthetic_df.columns = feature_cols + ["Attack_enc"]

        # Restore the original class label
        synthetic_df[target_col] = attack_label
        synthetic_df = synthetic_df[feature_cols + [target_col]].copy()

        # Resample to the desired class size
        synthetic_df = synthetic_df.sample(
            n=n_target,
            replace=len(synthetic_df) < n_target,
            random_state=42
        ).reset_index(drop=True)

        # Save class-specific output
        class_output_path = output_dir / f"synthetic_dataset_tabgan_{cfg['name']}_{attack_label}.csv"
        synthetic_df.to_csv(class_output_path, index=False)

        print(f"Saved class-specific file: {class_output_path}")
        print(synthetic_df[target_col].value_counts())

        class_outputs.append(synthetic_df)

        summary_rows.append({
            "variant": cfg["name"],
            "class": attack_label,
            "requested_rows": n_target,
            "generated_rows": len(synthetic_df),
            "output_path": str(class_output_path)
        })

    # Merge all class-specific outputs into one dataset for this variant
    merged_df = pd.concat(class_outputs, ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

    merged_output_path = output_dir / f"synthetic_dataset_tabgan_{cfg['name']}_merged.csv"
    merged_df.to_csv(merged_output_path, index=False)

    print(f"\nSaved merged variant file: {merged_output_path}")
    print(merged_df[target_col].value_counts())

# --------------------------------------------------
# Save summary table
# --------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_output_path = output_dir / "tabgan_classwise_variant_summary.csv"
summary_df.to_csv(summary_output_path, index=False)

print(f"\nSaved summary file: {summary_output_path}")
print(summary_df)

Target counts: {'none': 10053, 'Backdoor': 14796, 'syn-flood': 9463}

Running TabGAN variant: best_variant_1
Configuration: {'name': 'best_variant_1', 'gen_x_times': 0.75, 'pregeneration_frac': 1, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.9999, 'use_adversarial': False, 'gen_params': {'batch_size': 300, 'patience': 25, 'epochs': 100}}

Training separate TabGAN for class: none


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_best_variant_1_none.csv
Attack
none    10053
Name: count, dtype: int64

Training separate TabGAN for class: Backdoor


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

...

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_best_variant_1_Backdoor.csv
Attack
Backdoor    14796
Name: count, dtype: int64

Training separate TabGAN for class: syn-flood


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/100 [00:00<?, ?it/s]

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_best_variant_1_syn-flood.csv
Attack
syn-flood    9463
Name: count, dtype: int64

Saved merged variant file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_best_variant_1_merged.csv
Attack
Backdoor     14796
none         10053
syn-flood     9463
Name: count, dtype: int64



TypeError: string indices must be integers, not 'str'

In [4]:
from pathlib import Path
import pandas as pd
from tabgan.sampler import GANGenerator

# --------------------------------------------------
# Paths
# --------------------------------------------------
train_path = Path("../datasets/CICEVSE2024/synthetic_extension/train_dataset.csv")
output_dir = Path("../datasets/CICEVSE2024/synthetic_extension")
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Dataset configuration
# State is treated as a single binary feature.
# It remains in the feature set so it can influence
# the generated samples, but it is not one-hot encoded.
# --------------------------------------------------
feature_cols = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]
target_col = "Attack"
cat_cols = ["State"]

target_total_rows = 34312
target_distribution = {
    "none": 0.2930,
    "Backdoor": 0.4312,
    "syn-flood": 0.2758
}

# --------------------------------------------------
# TabGAN variants
# --------------------------------------------------
variant_configs = [
    {
        "name": "baseline",
        "gen_x_times": 1.0,
        "pregeneration_frac": 2,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 300},
    },
    {
        "name": "variant_a",
        "gen_x_times": 1.5,
        "pregeneration_frac": 2,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 200},
    },
    {
        "name": "variant_b",
        "gen_x_times": 2.0,
        "pregeneration_frac": 3,
        "is_post_process": True,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 25, "epochs": 300},
    },
    {
        "name": "variant_c",
        "gen_x_times": 1.5,
        "pregeneration_frac": 3,
        "is_post_process": False,
        "bot_filter_quantile": 0.001,
        "top_filter_quantile": 0.999,
        "use_adversarial": False,
        "gen_params": {"batch_size": 250, "patience": 20, "epochs": 300},
    },
]

# --------------------------------------------------
# Load and validate dataset
# --------------------------------------------------
df = pd.read_csv(train_path)
required_cols = feature_cols + [target_col]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy()
df["Attack"] = df["Attack"].astype(str)

# --------------------------------------------------
# Encode State as a single binary column
# Replace the mapping below with the exact two values
# that occur in your dataset if needed.
# --------------------------------------------------
unique_states = sorted(df["State"].astype(str).unique())
if len(unique_states) != 2:
    raise ValueError(
        f"State must be binary, but found {len(unique_states)} unique values: {unique_states}"
    )

state_map = {
    unique_states[0]: 0,
    unique_states[1]: 1
}

df["State"] = df["State"].astype(str).map(state_map).astype(int)

print("State mapping used:", state_map)
print("State value counts after binary encoding:")
print(df["State"].value_counts())

# --------------------------------------------------
# Compute target row count per class
# --------------------------------------------------
target_counts = {k: int(target_total_rows * v) for k, v in target_distribution.items()}
remainder = target_total_rows - sum(target_counts.values())

if remainder != 0:
    largest_class = max(target_distribution, key=target_distribution.get)
    target_counts[largest_class] += remainder

print("Target counts:", target_counts)

# --------------------------------------------------
# Run class-wise TabGAN generation
# Each class is trained independently with its own model.
# --------------------------------------------------
summary_rows = []

for cfg in variant_configs:
    print(f"\n{'=' * 70}")
    print(f"Running TabGAN variant: {cfg['name']}")
    print(f"Configuration: {cfg}")
    print(f"{'=' * 70}")

    class_outputs = []

    for attack_label, n_target in target_counts.items():
        print(f"\nTraining separate TabGAN for class: {attack_label}")

        # Select only rows from the current class
        df_class = df[df[target_col] == attack_label].copy()

        if df_class.empty:
            raise ValueError(f"No rows found for class '{attack_label}'.")

        if len(df_class) < 10:
            raise ValueError(f"Not enough rows for class '{attack_label}' to train a stable TabGAN.")

        # A separate model is trained for each class.
        # The class label is therefore constant inside this subset.
        df_class["Attack_enc"] = 0

        train_df = df_class[feature_cols].copy()
        target_df = df_class[["Attack_enc"]].copy()

        # Use the class-specific feature distribution as reference
        test_df = train_df.copy()

        generator = GANGenerator(
            gen_x_times=cfg["gen_x_times"],
            cat_cols=cat_cols,
            pregeneration_frac=cfg["pregeneration_frac"],
            is_post_process=cfg["is_post_process"],
            bot_filter_quantile=cfg["bot_filter_quantile"],
            top_filter_quantile=cfg["top_filter_quantile"],
            gen_params=cfg["gen_params"]
        )

        synthetic_x, synthetic_y = generator.generate_data_pipe(
            train_df=train_df,
            target=target_df,
            test_df=test_df,
            deep_copy=True,
            only_adversarial=False,
            use_adversarial=cfg["use_adversarial"]
        )

        synthetic_df = pd.concat(
            [synthetic_x.reset_index(drop=True), synthetic_y.reset_index(drop=True)],
            axis=1
        )

        # Ensure target column naming is correct
        if "Attack_enc" not in synthetic_df.columns:
            if synthetic_df.shape[1] == len(feature_cols) + 1:
                synthetic_df.columns = feature_cols + ["Attack_enc"]

        # Force State back to binary integer format after generation
        # This avoids decimal drift in a feature that should stay binary.
        synthetic_df["State"] = (synthetic_df["State"] >= 0.5).astype(int)

        # Restore the original class label
        synthetic_df[target_col] = attack_label
        synthetic_df = synthetic_df[feature_cols + [target_col]].copy()

        # Resample to the desired class size
        synthetic_df = synthetic_df.sample(
            n=n_target,
            replace=len(synthetic_df) < n_target,
            random_state=42
        ).reset_index(drop=True)

        # Save class-specific output
        class_output_path = output_dir / f"synthetic_dataset_tabgan_{cfg['name']}_{attack_label}.csv"
        synthetic_df.to_csv(class_output_path, index=False)

        print(f"Saved class-specific file: {class_output_path}")
        print(synthetic_df[target_col].value_counts())
        print("State distribution:")
        print(synthetic_df["State"].value_counts())

        class_outputs.append(synthetic_df)

        summary_rows.append({
            "variant": cfg["name"],
            "class": attack_label,
            "requested_rows": n_target,
            "generated_rows": len(synthetic_df),
            "output_path": str(class_output_path)
        })

    # Merge all class-specific outputs into one dataset for this variant
    merged_df = pd.concat(class_outputs, ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

    merged_output_path = output_dir / f"synthetic_dataset_tabgan_{cfg['name']}_merged.csv"
    merged_df.to_csv(merged_output_path, index=False)

    print(f"\nSaved merged variant file: {merged_output_path}")
    print(merged_df[target_col].value_counts())
    print("Merged State distribution:")
    print(merged_df["State"].value_counts())

# --------------------------------------------------
# Save summary table
# --------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_output_path = output_dir / "tabgan_classwise_variant_summary.csv"
summary_df.to_csv(summary_output_path, index=False)

print(f"\nSaved summary file: {summary_output_path}")
print(summary_df)

State mapping used: {'charging': 0, 'idle': 1}
State value counts after binary encoding:
State
1    19491
0    14820
Name: count, dtype: int64
Target counts: {'none': 10053, 'Backdoor': 14796, 'syn-flood': 9463}

Running TabGAN variant: baseline
Configuration: {'name': 'baseline', 'gen_x_times': 1.0, 'pregeneration_frac': 2, 'is_post_process': True, 'bot_filter_quantile': 0.001, 'top_filter_quantile': 0.999, 'use_adversarial': False, 'gen_params': {'batch_size': 250, 'patience': 20, 'epochs': 300}}

Training separate TabGAN for class: none


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/300 [00:00<?, ?it/s]

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_baseline_none.csv
Attack
none    10053
Name: count, dtype: int64
State distribution:
State
1    5140
0    4913
Name: count, dtype: int64

Training separate TabGAN for class: Backdoor


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

...

Training CTGAN, epochs::   0%|          | 0/300 [00:00<?, ?it/s]

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_variant_c_Backdoor.csv
Attack
Backdoor    14796
Name: count, dtype: int64
State distribution:
State
0    9776
1    5020
Name: count, dtype: int64

Training separate TabGAN for class: syn-flood


Fitting CTGAN transformers for each column:   0%|          | 0/6 [00:00<?, ?it/s]

Training CTGAN, epochs::   0%|          | 0/300 [00:00<?, ?it/s]

Saved class-specific file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_variant_c_syn-flood.csv
Attack
syn-flood    9463
Name: count, dtype: int64
State distribution:
State
1    9463
Name: count, dtype: int64

Saved merged variant file: ../datasets/CICEVSE2024/synthetic_extension/synthetic_dataset_tabgan_variant_c_merged.csv
Attack
Backdoor     14796
none         10053
syn-flood     9463
Name: count, dtype: int64
Merged State distribution:
State
1    19650
0    14662
Name: count, dtype: int64

Saved summary file: ../datasets/CICEVSE2024/synthetic_extension/tabgan_classwise_variant_summary.csv
      variant      class  requested_rows  generated_rows  \
0    baseline       none           10053           10053   
1    baseline   Backdoor           14796           14796   
2    baseline  syn-flood            9463            9463   
3   variant_a       none           10053           10053   
4   variant_a   Backdoor           14796           14796   
5   variant_a  s